In [1]:


import numpy as np
import re
import requests
from scipy import stats
from datetime import datetime
import zipfile
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import scipy.stats as stats
from scipy.optimize import curve_fit
import shutil
from matplotlib import font_manager, rc
import subprocess
import sys
import os
import random
import glob
import time 
import folium
from folium.plugins import HeatMap
from tqdm import tqdm
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import folium

from geopy.distance import geodesic
import pandas as pd
import json

In [2]:

# CSV 파일 경로와 이름(경도와 위도에 대한 정보가 들어 있는 csv, 다운 받은 경로로 변경)
file_path = "/home/addd/Downloads/seoul_2023_09.csv"

df= pd.read_csv(file_path)

/tmp/ipykernel_45143/3082750892.py:4: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  df= pd.read_csv(file_path)


In [3]:
# 대분류별 중분류 구하기
중분류_dict = {}  # 중분류를 저장할 딕셔너리 초기화
# 대분류 추출 및 리스트 저장
대분류 = list(df['상권업종대분류명'].unique())

# 대분류별 중분류별 소분류 구하기
소분류_dict = {}  # 소분류를 저장할 딕셔너리 초기화

for 대분류명 in 대분류:
    중분류_dict = {}  # 중분류를 저장할 딕셔너리 초기화
    중분류 = list(df[df['상권업종대분류명'] == 대분류명]['상권업종중분류명'].unique())
    for 중분류명 in 중분류:
        소분류 = list(df[(df['상권업종대분류명'] == 대분류명) & (df['상권업종중분류명'] == 중분류명)]['상권업종소분류명'].unique())
        중분류_dict[중분류명] = 소분류
    소분류_dict[대분류명] = 중분류_dict

# 결과 출력
for 대분류명, 중분류_dict in 소분류_dict.items():
    print("대분류명:", 대분류명)
    for 중분류명, 소분류 in 중분류_dict.items():
        print("  중분류명:", 중분류명)
        print("  소분류명:", 소분류)
    print()


대분류명: 음식
  중분류명: 한식
  소분류명: ['곱창 전골/구이', '백반/한정식', '닭/오리고기 구이/찜', '돼지고기 구이/찜', '족발/보쌈', '국수/칼국수', '국/탕/찌개류', '냉면/밀면', '해산물 구이/찜', '횟집', '전/부침개', '소고기 구이/찜', '기타 한식 음식점', '복 요리 전문']
  중분류명: 기타 간이
  소분류명: ['치킨', '피자', '빵/도넛', '떡/한과', '김밥/만두/분식', '토스트/샌드위치/샐러드', '버거', '그 외 기타 간이 음식점', '아이스크림/빙수']
  중분류명: 비알코올 
  소분류명: ['카페']
  중분류명: 주점
  소분류명: ['생맥주 전문', '요리 주점', '일반 유흥 주점', '무도 유흥 주점']
  중분류명: 동남아시아
  소분류명: ['베트남식 전문', '기타 동남아식 전문']
  중분류명: 서양식
  소분류명: ['경양식', '기타 서양식 음식점', '파스타/스테이크', '패밀리레스토랑']
  중분류명: 중식
  소분류명: ['중국집', '마라탕/훠궈']
  중분류명: 구내식당·뷔페
  소분류명: ['구내식당', '뷔페']
  중분류명: 일식
  소분류명: ['일식 회/초밥', '일식 카레/돈가스/덮밥', '일식 면 요리', '기타 일식 음식점']
  중분류명: 기타 외국
  소분류명: ['분류 안된 외국식 음식점']

대분류명: 숙박
  중분류명: 일반 숙박
  소분류명: ['여관/모텔', '펜션', '호텔/리조트', '캠핑/글램핑']
  중분류명: 기타 숙박
  소분류명: ['기숙사/고시원', '그 외 기타 숙박업']

대분류명: 교육
  중분류명: 기타 교육
  소분류명: ['태권도/무술학원', '기타 기술/직업 훈련학원', '레크리에이션 교육기관', '요가/필라테스 학원', '기타 예술/스포츠 교육기관', '그 외 기타 교육기관', '운전학원', '외국어학원', '미술학원', '음악학원', '사회교육시설', '직원 훈련기관', '컴퓨터 학원', '청소년 수련시설

In [4]:
시군구명_list = list(df['시군구명'].unique())  # 시군구명 리스트 추출

시군구_dict = {}  # 시군구명을 부모로 하는 딕셔너리 초기화

for 시군구명 in 시군구명_list:
    법정동_list = list(df[df['시군구명'] == 시군구명]['법정동명'].unique())
    시군구_dict[시군구명] = 법정동_list

# 결과 출력
for 시군구명, 법정동_list in 시군구_dict.items():
    print("시군구명:", 시군구명)
    print("  법정동명:", 법정동_list)
    print()


시군구명: 광진구
  법정동명: ['중곡동', '자양동', '군자동', '화양동', '구의동', '광장동', '능동']

시군구명: 중구
  법정동명: ['광희동1가', '중림동', '신당동', '남대문로2가', '필동3가', '장교동', '서소문동', '만리동1가', '필동2가', '황학동', '남대문로3가', '회현동1가', '을지로7가', '순화동', '충무로2가', '수표동', '을지로6가', '광희동2가', '저동1가', '묵정동', '예관동', '산림동', '남대문로5가', '무교동', '필동1가', '인현동2가', '오장동', '초동', '의주로2가', '다동', '을지로2가', '소공동', '저동2가', '충무로1가', '남창동', '방산동', '을지로3가', '충무로4가', '장충동2가', '태평로1가', '태평로2가', '명동2가', '회현동2가', '정동', '주교동', '흥인동', '을지로1가', '남산동2가', '남산동1가', '을지로4가', '회현동3가', '충무로3가', '충무로5가', '입정동', '북창동', '장충동1가', '을지로5가', '무학동', '남대문로4가', '예장동', '인현동1가', '수하동', '쌍림동', '삼각동', '명동1가', '만리동2가', '봉래동2가', '봉래동1가', '의주로1가', '남산동3가', '남대문로1가', '남학동', '충정로1가', '주자동']

시군구명: 노원구
  법정동명: ['중계동', '상계동', '공릉동', '하계동', '월계동']

시군구명: 양천구
  법정동명: ['신월동', '신정동', '목동']

시군구명: 강서구
  법정동명: ['마곡동', '염창동', '화곡동', '등촌동', '방화동', '내발산동', '공항동', '가양동', '외발산동', '과해동', '개화동', '오쇠동', '오곡동']

시군구명: 송파구
  법정동명: ['송파동', '문정동', '장지동', '거여동', '석촌동', '풍납동', '가락동', '잠실동', '방이동', '오금동', '마천동', '삼전동'

In [5]:


# 시군구명과 법정동명을 저장한 딕셔너리를 JSON 파일로 저장
with open('/home/addd/Desktop/시군구_dict.json', 'w', encoding='utf-8') as f:
    json.dump(시군구_dict, f, ensure_ascii=False, indent=4)

# 대분류, 중분류, 소분류를 저장한 딕셔너리를 JSON 파일로 저장
with open('/home/addd/Desktop/소분류_dict.json', 'w', encoding='utf-8') as f:
    json.dump(소분류_dict, f, ensure_ascii=False, indent=4)


In [6]:
print(소분류_dict)

{'음식': {'한식': ['곱창 전골/구이', '백반/한정식', '닭/오리고기 구이/찜', '돼지고기 구이/찜', '족발/보쌈', '국수/칼국수', '국/탕/찌개류', '냉면/밀면', '해산물 구이/찜', '횟집', '전/부침개', '소고기 구이/찜', '기타 한식 음식점', '복 요리 전문'], '기타 간이': ['치킨', '피자', '빵/도넛', '떡/한과', '김밥/만두/분식', '토스트/샌드위치/샐러드', '버거', '그 외 기타 간이 음식점', '아이스크림/빙수'], '비알코올 ': ['카페'], '주점': ['생맥주 전문', '요리 주점', '일반 유흥 주점', '무도 유흥 주점'], '동남아시아': ['베트남식 전문', '기타 동남아식 전문'], '서양식': ['경양식', '기타 서양식 음식점', '파스타/스테이크', '패밀리레스토랑'], '중식': ['중국집', '마라탕/훠궈'], '구내식당·뷔페': ['구내식당', '뷔페'], '일식': ['일식 회/초밥', '일식 카레/돈가스/덮밥', '일식 면 요리', '기타 일식 음식점'], '기타 외국': ['분류 안된 외국식 음식점']}, '숙박': {'일반 숙박': ['여관/모텔', '펜션', '호텔/리조트', '캠핑/글램핑'], '기타 숙박': ['기숙사/고시원', '그 외 기타 숙박업']}, '교육': {'기타 교육': ['태권도/무술학원', '기타 기술/직업 훈련학원', '레크리에이션 교육기관', '요가/필라테스 학원', '기타 예술/스포츠 교육기관', '그 외 기타 교육기관', '운전학원', '외국어학원', '미술학원', '음악학원', '사회교육시설', '직원 훈련기관', '컴퓨터 학원', '청소년 수련시설', '전문자격/고시학원'], '교육 지원': ['기타 교육지원 서비스업', '교육컨설팅업'], '일반 교육': ['입시·교과학원']}, '과학·기술': {'사진 촬영': ['사진촬영업'], '광고': ['광고 대행업', '광고물 설계/제작업', '옥외/전시 광고 대행업', '기타 광고 관

In [9]:


def selected_middle_classification(low_classification, korea_area, radius):
    # '상권업종중분류명'이 입력값과 일치하고 '법정동명'이 입력한 지역구와 일치하는 행만 선택합니다
    df_selected = df[(df['상권업종소분류명'] == low_classification) & (df['법정동명'] == korea_area)]

    # 입력한 지역구의 중심 좌표를 가져옵니다
    center_latitude = df_selected['위도'].mean()
    center_longitude = df_selected['경도'].mean()

    # 지도를 생성합니다. 초기 위치는 입력한 지역구의 중심 좌표로 설정합니다
    m = folium.Map(location=[center_latitude, center_longitude], zoom_start=13)
    print(center_latitude, center_longitude)
    # 선택된 행들의 각 위도와 경도에 마커를 추가합니다
    for idx, row in df_selected.iterrows():
        # 입력한 반경 내에 있는 상호만 선택하여 마커를 추가합니다
        distance = geodesic((center_latitude, center_longitude), (row['위도'], row['경도'])).km
        if distance <= radius:
            # 팝업에 표시될 정보를 설정합니다
            popup_info = f"상호명: {row['상호명']}<br>법정동명: {row['법정동명']}<br>도로명주소: {row['도로명주소']}"
            folium.Marker([row['위도'], row['경도']], popup=folium.Popup(popup_info, max_width=750)).add_to(m)

    # HTML 파일로 저장합니다. 파일명은 'map_class.html'입니다.
    m.save('map_class.html')

# 함수 호출 예시
low_classification = '국수/칼국수'
korea_area = '역삼동'
radius = 0.3
selected_middle_classification(low_classification, korea_area, radius)


37.4996595609691 127.03767637700433
